# 01 — Setup S3 Data Lake for Yelp Sentiment MLOps

This notebook is the entry point for the AWS portion of the Yelp Review Sentiment MLOps project.

By the end of this notebook you will have:

1. Verified your SageMaker / AWS identity.
2. Created a dedicated S3 bucket for the Yelp sentiment project.
3. Uploaded the raw Yelp Open Dataset (`Yelp-JSON.zip`) into the bucket under `raw/`.
4. Streamed the review JSON out of the nested zip/tar archive and written a sample of raw reviews to CSV.
5. Uploaded the raw-review CSV to S3 under `raw/reviews/` so that Athena and the next notebooks can query it.

This mirrors the structure of the AAI-508 Heart Valve project's `01_setup_S3_bucket.ipynb`.

In [ ]:
import boto3
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]

print("Region:    ", region)
print("Account ID:", account_id)
print("Role:      ", role)

## Define the bucket and prefixes

We use a deterministic bucket name that includes your AWS account ID so the bucket is globally unique. The prefix layout below mirrors what you will see in the `docs/aws_mlops_plan.md` document and follows the same `raw/`, `processed/`, `features/`, `models/`, `reports/` convention used in the Heart Valve project.

In [ ]:
bucket = f"yelp-sentiment-mlops-{account_id}"
raw_prefix = "raw"
raw_zip_key = f"{raw_prefix}/yelp-json.zip"
raw_reviews_prefix = f"{raw_prefix}/reviews"
athena_staging_prefix = "athena/staging"

print("Bucket:                 ", bucket)
print("Raw zip key:            ", raw_zip_key)
print("Raw reviews CSV prefix: ", raw_reviews_prefix)
print("Athena staging prefix:  ", athena_staging_prefix)

%store bucket
%store raw_prefix
%store raw_zip_key
%store raw_reviews_prefix
%store athena_staging_prefix
%store region
%store account_id

## Create the S3 bucket if it does not exist

In [ ]:
s3 = boto3.client("s3", region_name=region)

def ensure_bucket(bucket_name: str, region_name: str) -> None:
    existing = {b["Name"] for b in s3.list_buckets().get("Buckets", [])}
    if bucket_name in existing:
        print(f"Bucket already exists: {bucket_name}")
        return
    if region_name == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region_name},
        )
    print(f"Created bucket: {bucket_name}")

ensure_bucket(bucket, region)

## Upload `Yelp-JSON.zip` to S3 (one-time, from your local machine)

The full Yelp Open Dataset is too large to ship through GitHub (4 GB compressed). You only need to upload it **once** from your local machine using the AWS CLI:

```bash
# Run this on your local laptop, not inside SageMaker
aws s3 cp data/Yelp-JSON.zip s3://<bucket-name-printed-above>/raw/yelp-json.zip
```

The cell below verifies the upload completed before continuing. If it has not been uploaded yet, run the command on your laptop and re-run this cell.

In [ ]:
from botocore.exceptions import ClientError

try:
    head = s3.head_object(Bucket=bucket, Key=raw_zip_key)
    size_gb = head["ContentLength"] / (1024 ** 3)
    print(f"Found s3://{bucket}/{raw_zip_key} ({size_gb:.2f} GB)")
except ClientError as exc:
    print("NOT FOUND yet. Upload from your laptop with:")
    print(f"  aws s3 cp data/Yelp-JSON.zip s3://{bucket}/{raw_zip_key}")
    raise

## Stream the review JSON Lines out of the nested zip/tar archive

The Yelp Open Dataset archive is structured as:

```
Yelp-JSON.zip
  └── Yelp JSON/yelp_dataset.tar
          └── yelp_academic_dataset_review.json   (JSON Lines)
```

We stream straight from S3 → ZipFile → TarFile → JSON Lines so we never need to expand the entire 9 GB review file on disk. The `MAX_RAW_REVIEWS` setting controls how many lines we pull; 300,000 raw reviews comfortably exceeds the rubric requirement of 10,000 records per class once neutral 3-star reviews are filtered out and the classes are balanced in notebook 03.

In [ ]:
import csv
import io
import json
import os
import tarfile
import zipfile
from pathlib import Path

MAX_RAW_REVIEWS = int(os.environ.get("YELP_MAX_REVIEWS", "300000"))
REVIEW_MEMBER_SUFFIX = "yelp_academic_dataset_review.json"

LOCAL_DATA_DIR = Path("/home/sagemaker-user/yelp-sentiment-mlops-pipeline/data")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ZIP_PATH = LOCAL_DATA_DIR / "Yelp-JSON.zip"
LOCAL_CSV_PATH = LOCAL_DATA_DIR / "reviews_raw.csv"

if not LOCAL_ZIP_PATH.exists():
    print(f"Downloading s3://{bucket}/{raw_zip_key} to {LOCAL_ZIP_PATH} ...")
    s3.download_file(bucket, raw_zip_key, str(LOCAL_ZIP_PATH))
    print("Download complete.")
else:
    print(f"Reusing existing {LOCAL_ZIP_PATH}")

print(f"Will extract up to {MAX_RAW_REVIEWS:,} raw reviews.")

In [ ]:
CSV_COLUMNS = ["review_id", "business_id", "user_id", "stars", "review_text", "date"]

def normalize_review(review: dict) -> dict:
    return {
        "review_id": review.get("review_id", ""),
        "business_id": review.get("business_id", ""),
        "user_id": review.get("user_id", ""),
        "stars": review.get("stars", ""),
        "review_text": (review.get("text") or "").replace("\n", " ").replace("\r", " "),
        "date": review.get("date", ""),
    }

def stream_reviews_to_csv(zip_path: Path, csv_path: Path, max_reviews: int) -> int:
    written = 0
    with zipfile.ZipFile(zip_path) as zf:
        tar_name = next(
            name for name in zf.namelist()
            if name.endswith(".tar") and not name.startswith("__MACOSX/")
        )
        with zf.open(tar_name) as tar_stream:
            with tarfile.open(fileobj=tar_stream, mode="r|gz") as tf:
                with csv_path.open("w", newline="", encoding="utf-8") as out_file:
                    writer = csv.DictWriter(out_file, fieldnames=CSV_COLUMNS)
                    writer.writeheader()
                    for member in tf:
                        if not member.name.endswith(REVIEW_MEMBER_SUFFIX):
                            continue
                        extracted = tf.extractfile(member)
                        if extracted is None:
                            break
                        for line in extracted:
                            review = json.loads(line.decode("utf-8"))
                            writer.writerow(normalize_review(review))
                            written += 1
                            if written % 25000 == 0:
                                print(f"  wrote {written:,} reviews so far")
                            if written >= max_reviews:
                                return written
                        break
    return written

row_count = stream_reviews_to_csv(LOCAL_ZIP_PATH, LOCAL_CSV_PATH, MAX_RAW_REVIEWS)
print(f"Wrote {row_count:,} raw reviews to {LOCAL_CSV_PATH}")

## Upload the raw-reviews CSV to S3

Athena will register an external table over this CSV in the next notebook.

In [ ]:
raw_reviews_key = f"{raw_reviews_prefix}/reviews_raw.csv"
s3.upload_file(str(LOCAL_CSV_PATH), bucket, raw_reviews_key)
print(f"Uploaded to s3://{bucket}/{raw_reviews_key}")
%store raw_reviews_key

In [ ]:
print(f"S3 contents under s3://{bucket}/:")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket):
    for obj in page.get("Contents", []):
        size_mb = obj["Size"] / (1024 ** 2)
        print(f"  {obj['Key']:<60} {size_mb:>10.2f} MB")

## Done

Continue to `athena_queries/01_Create_Athena_Database.ipynb`.